# Benchmark Machine Learning — MaroTrade Intelligence

**Objectif :** Comparer XGBoost, LightGBM et CatBoost pour identifier le meilleur modèle de scoring des marchés export.

**Pipeline :**
1. Chargement et exploration des données
2. Preprocessing et feature engineering
3. Construction de la variable cible
4. Benchmark des modèles
5. Optimisation avec Optuna
6. Évaluation finale + SHAP
7. Conclusion et recommandation

## 0. Installation des dépendances

In [2]:
# Installation
import subprocess
packages = ['lightgbm', 'catboost', 'optuna', 'mlflow', 'shap', 'plotly']
for pkg in packages:
    subprocess.run(['pip', 'install', pkg, '-q'], capture_output=True)
print('✅ Packages installés')

✅ Packages installés


## 1. Imports

In [5]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
import time
from pathlib import Path

# ML
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# SHAP + Optuna
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# MLflow
import mlflow
import mlflow.sklearn

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Config
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')
RANDOM_STATE = 42
DATA_DIR = Path('../data/raw')

print('✅ Imports OK')

✅ Imports OK


## 2. Chargement des données

In [6]:
# ── Données commerciales UN Comtrade ─────────────────────────
df_trade = pd.read_csv(DATA_DIR / 'marotrade_dataset.csv')
print(f'✅ Trade data     : {df_trade.shape[0]:,} lignes × {df_trade.shape[1]} colonnes')

# ── Indicateurs World Bank ────────────────────────────────────
df_wb = pd.read_csv(DATA_DIR / 'worldbank_indicators.csv', index_col=0)
print(f'✅ World Bank     : {df_wb.shape[0]} pays × {df_wb.shape[1]} indicateurs')

# ── Accords commerciaux ───────────────────────────────────────
df_accords = pd.read_csv(DATA_DIR / 'accords_maroc.csv', index_col=0)
print(f'✅ Accords Maroc  : {df_accords.shape[0]} pays')

# ── Risque OCDE ───────────────────────────────────────────────
df_ocde = pd.read_csv(DATA_DIR / 'ocde_risk.csv', index_col=0)
print(f'✅ OCDE Risque    : {df_ocde.shape[0]} pays')

# ── Google Trends ─────────────────────────────────────────────
with open(DATA_DIR / 'google_trends.json') as f:
    trends_raw = json.load(f)
print(f'✅ Google Trends  : {len(trends_raw)} produits')

✅ Trade data     : 4,770 lignes × 11 colonnes
✅ World Bank     : 20 pays × 7 indicateurs
✅ Accords Maroc  : 20 pays
✅ OCDE Risque    : 20 pays
✅ Google Trends  : 10 produits


## 3. Exploration des données (EDA)

In [7]:
print('=== APERÇU TRADE DATA ===')
display(df_trade.head())
print()
print('=== TYPES ===')
print(df_trade.dtypes)
print()
print('=== VALEURS MANQUANTES ===')
print(df_trade.isnull().sum())
print()
print('=== STATISTIQUES ===')
display(df_trade.describe())

=== APERÇU TRADE DATA ===


,hs_code,hs_desc,year,country_code,country_name,value_usd,weight_kg,qty,price_usd_kg,value_lag1,growth_yoy
0,30353,"Fish; frozen, sardines (Sardina pilchardus, Sa...",2019,36,Australie,32271.8640,31016.0000,31016.0000,1.0405,NaN,NaN
1,30353,"Fish; frozen, sardines (Sardina pilchardus, Sa...",2020,36,Australie,39645.3660,42840.0000,42840.0000,0.9254,32271.8640,22.8481
2,30353,"Fish; frozen, sardines (Sardina pilchardus, Sa...",2021,36,Australie,21332.5170,21420.0000,21420.0000,0.9959,39645.3660,-46.1917
3,30353,"Fish; frozen, sardines (Sardina pilchardus, Sa...",2016,56,Belgique,90.0000,96.0000,96.0000,0.9375,NaN,NaN
4,30353,"Fish; frozen, sardines (Sardina pilchardus, Sa...",2017,56,Belgique,5527.7810,3959.0000,3959.0000,1.3963,90.0000,6041.9789



=== TYPES ===
hs_code           int64
hs_desc          object
year              int64
country_code      int64
country_name     object
value_usd       float64
weight_kg       float64
qty             float64
price_usd_kg    float64
value_lag1      float64
growth_yoy      float64
dtype: object

=== VALEURS MANQUANTES ===
hs_code           0
hs_desc           0
year              0
country_code      0
country_name      0
value_usd         0
weight_kg         0
qty               0
price_usd_kg      0
value_lag1      804
growth_yoy      804
dtype: int64

=== STATISTIQUES ===


,hs_code,year,country_code,value_usd,weight_kg,qty,price_usd_kg,value_lag1,growth_yoy
count,4770.0000,4770.0000,4770.0000,4770.0000,4770.0000,4770.0000,4770.0000,3966.0000,3966.0000
mean,450811.5310,2019.2335,460.6763,24833641.6470,12321946.0793,11990122.7685,47.5566,24582420.0904,141814.7697
std,321665.9470,2.5646,268.3408,81803884.3340,53430288.6696,53335737.6426,348.6543,79189382.9760,3532582.9038
min,30353.0000,2015.0000,36.0000,0.0010,0.0000,0.0000,0.0000,0.0010,-99.9999
25%,150420.0000,2017.0000,251.0000,38865.4485,3287.7550,1851.0000,1.3207,51441.7092,-31.7188
50%,310540.0000,2019.0000,410.0000,1050281.7240,169350.0000,111492.2000,6.4395,1183955.3035,10.9992
75%,853690.0000,2021.0000,724.0000,13135155.8393,4158761.8250,3766926.0775,25.7799,13842728.2228,102.1755
max,940199.0000,2023.0000,842.0000,1345816050.7790,1068634900.0000,1068634900.0000,19552.7200,1233482268.8690,168164665.1627


In [ ]:
# Distribution des valeurs export par pays
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Top pays par valeur
top_countries = df_trade.groupby('country_name')['value_usd'].sum().sort_values(ascending=False).head(15)
axes[0,0].barh(top_countries.index, top_countries.values / 1e9)
axes[0,0].set_title('Top 15 pays — Valeur export (Mrd USD)')
axes[0,0].set_xlabel('Mrd USD')

# Evolution temporelle
yearly = df_trade.groupby('year')['value_usd'].sum()
axes[0,1].plot(yearly.index, yearly.values / 1e9, marker='o', linewidth=2)
axes[0,1].set_title('Evolution exports Maroc 2015–2023')
axes[0,1].set_xlabel('Année')
axes[0,1].set_ylabel('Mrd USD')

# Distribution growth_yoy
growth_clean = df_trade['growth_yoy'].dropna()
growth_clean = growth_clean[(growth_clean > -100) & (growth_clean < 200)]
axes[1,0].hist(growth_clean, bins=50, edgecolor='white')
axes[1,0].set_title('Distribution croissance YoY (%)')
axes[1,0].set_xlabel('Croissance (%)')

# Top produits
top_hs = df_trade.groupby('hs_code')['value_usd'].sum().sort_values(ascending=False).head(15)
axes[1,1].barh(top_hs.index.astype(str), top_hs.values / 1e9)
axes[1,1].set_title('Top 15 produits HS — Valeur (Mrd USD)')
axes[1,1].set_xlabel('Mrd USD')

plt.tight_layout()
plt.savefig('../data/raw/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA overview sauvegardé')

## 4. Feature Engineering

In [ ]:
def build_feature_matrix(df_trade, df_wb, df_accords, df_ocde, trends_raw):
    """
    Construit la matrice de features complète.
    Fusionne toutes les sources de données.
    """
    # Agréger par pays (dernière année disponible)
    df_2022 = df_trade[df_trade['year'] == 2022].copy()
    
    # CAGR par pays et produit
    df_cagr = df_trade.sort_values(['hs_code', 'country_code', 'year'])
    df_start = df_cagr[df_cagr['year'] == 2015][['hs_code', 'country_code', 'value_usd']].rename(columns={'value_usd': 'val_2015'})
    df_end   = df_cagr[df_cagr['year'] == 2022][['hs_code', 'country_code', 'value_usd']].rename(columns={'value_usd': 'val_2022'})
    df_growth = df_start.merge(df_end, on=['hs_code', 'country_code'], how='inner')
    df_growth['cagr'] = ((df_growth['val_2022'] / df_growth['val_2015'].replace(0, np.nan)) ** (1/7) - 1) * 100
    df_growth['cagr'] = df_growth['cagr'].clip(-50, 100)
    
    # Momentum (accélération récente)
    df_2020 = df_trade[df_trade['year'] == 2020][['hs_code', 'country_code', 'value_usd']].rename(columns={'value_usd': 'val_2020'})
    df_growth = df_growth.merge(df_2020, on=['hs_code', 'country_code'], how='left')
    df_growth['cagr_recent'] = ((df_growth['val_2022'] / df_growth['val_2020'].replace(0, np.nan)) ** (1/2) - 1) * 100
    df_growth['momentum'] = df_growth['cagr_recent'].fillna(0) - df_growth['cagr'].fillna(0)
    
    # Fusionner avec données 2022
    df = df_2022.merge(df_growth[['hs_code', 'country_code', 'cagr', 'momentum']], 
                       on=['hs_code', 'country_code'], how='left')
    
    # World Bank
    wb_cols = ['ease_business', 'rule_of_law', 'reg_quality', 
               'political_stability', 'gdp_per_capita']
    wb_available = [c for c in wb_cols if c in df_wb.columns]
    df_wb_clean = df_wb[wb_available].copy()
    df_wb_clean.index.name = 'country_code'
    df_wb_clean = df_wb_clean.reset_index()
    df = df.merge(df_wb_clean, on='country_code', how='left')
    
    # Accords commerciaux
    acc_cols = ['droits', 'type']
    acc_available = [c for c in acc_cols if c in df_accords.columns]
    df_acc = df_accords[acc_available].copy()
    df_acc.index.name = 'country_code'
    df_acc = df_acc.reset_index()
    df = df.merge(df_acc, on='country_code', how='left')
    
    # OCDE risque
    if 'category' in df_ocde.columns:
        df_ocde_clean = df_ocde[['category', 'score']].copy()
        df_ocde_clean.index.name = 'country_code'
        df_ocde_clean = df_ocde_clean.reset_index()
        df_ocde_clean.columns = ['country_code', 'risk_category', 'risk_score']
        df = df.merge(df_ocde_clean, on='country_code', how='left')
    
    # Google Trends
    geo_map = {'FRA': 'FR', 'DEU': 'DE', 'USA': 'US', 'ESP': 'ES',
               'GBR': 'GB', 'JPN': 'JP', 'SAU': 'SA', 'ARE': 'AE'}
    
    def get_trend(row):
        hs   = str(row['hs_code'])
        geo  = geo_map.get(str(row['country_code']), None)
        if hs in trends_raw and geo and geo in trends_raw[hs]:
            return trends_raw[hs][geo].get('mean', 50)
        return 50
    
    df['trend_score'] = df.apply(get_trend, axis=1)
    
    # Encoder accord type
    if 'type' in df.columns:
        type_map = {'ALE': 1.0, 'PREF': 0.6, 'NPF': 0.3}
        df['accord_score'] = df['type'].map(type_map).fillna(0.3)
    
    # Prix moyen
    df['price_per_kg'] = df['value_usd'] / (df['weight_kg'].replace(0, np.nan))
    df['price_per_kg'] = df['price_per_kg'].fillna(df['price_per_kg'].median())
    
    print(f'✅ Matrice construite : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
    return df

df_features = build_feature_matrix(df_trade, df_wb, df_accords, df_ocde, trends_raw)
display(df_features.head())

In [ ]:
# ── Variable cible : score pondéré ────────────────────────────
def compute_target_score(df):
    """
    Calcule le score cible (0-100) basé sur les 7 dimensions.
    C'est ce que XGBoost doit apprendre à prédire.
    """
    scaler = MinMaxScaler()
    
    # Normaliser les features clés
    def norm(series):
        vals = series.fillna(series.median()).values.reshape(-1, 1)
        return scaler.fit_transform(vals).flatten()
    
    def norm_inv(series):
        return 1 - norm(series)
    
    # Dimensions
    dim_marche    = (norm(df['value_usd']) * 0.5 + 
                     norm(df['cagr'].fillna(0)) * 0.35 + 
                     norm(df['trend_score']) * 0.15)
    
    dim_accord    = (df.get('accord_score', pd.Series(0.3, index=df.index)) * 0.6 + 
                     norm_inv(df.get('droits', pd.Series(5.0, index=df.index))) * 0.4)
    
    dim_business  = (norm(df.get('ease_business', pd.Series(70, index=df.index))) * 0.35 +
                     norm(df.get('rule_of_law', pd.Series(70, index=df.index))) * 0.35 +
                     norm(df.get('reg_quality', pd.Series(70, index=df.index))) * 0.30)
    
    dim_stabilite = (norm(df.get('political_stability', pd.Series(60, index=df.index))) * 0.5 +
                     norm(df.get('risk_score', pd.Series(80, index=df.index))) * 0.5)
    
    dim_momentum  = norm(df['momentum'].fillna(0))
    
    # Score final pondéré
    score = (
        dim_marche    * 0.28 +
        dim_accord    * 0.22 +
        dim_business  * 0.18 +
        dim_stabilite * 0.12 +
        dim_momentum  * 0.10 +
        norm(df['trend_score']) * 0.10
    ) * 100
    
    return score

df_features['target_score'] = compute_target_score(df_features)
print(f'✅ Variable cible calculée')
print(f'   Min  : {df_features["target_score"].min():.2f}')
print(f'   Max  : {df_features["target_score"].max():.2f}')
print(f'   Mean : {df_features["target_score"].mean():.2f}')

# Distribution
plt.figure(figsize=(10, 4))
plt.hist(df_features['target_score'], bins=50, edgecolor='white', color='steelblue')
plt.title('Distribution du score cible (0-100)')
plt.xlabel('Score')
plt.ylabel('Fréquence')
plt.tight_layout()
plt.show()

## 5. Préparation train/test

In [ ]:
# Sélection des features finales
FEATURE_COLS = [
    'value_usd', 'cagr', 'momentum', 'price_per_kg',
    'accord_score', 'droits',
    'ease_business', 'rule_of_law', 'reg_quality', 'political_stability',
    'risk_score', 'risk_category',
    'trend_score', 'gdp_per_capita',
]

# Garder uniquement les colonnes disponibles
available_cols = [c for c in FEATURE_COLS if c in df_features.columns]
print(f'Features disponibles : {len(available_cols)}/{len(FEATURE_COLS)}')
print(available_cols)

# Dataset propre
df_clean = df_features[available_cols + ['target_score']].copy()
df_clean = df_clean.dropna(subset=['target_score'])
df_clean[available_cols] = df_clean[available_cols].fillna(df_clean[available_cols].median())

X = df_clean[available_cols].values
y = df_clean['target_score'].values / 100  # Normaliser 0-1

# Augmentation ×5 avec bruit gaussien
np.random.seed(RANDOM_STATE)
X_aug = np.vstack([X + np.random.normal(0, 0.02, X.shape) for _ in range(4)] + [X])
y_aug = np.tile(y, 5)

# Split 85/15
X_train, X_test, y_train, y_test = train_test_split(
    X_aug, y_aug, test_size=0.15, random_state=RANDOM_STATE
)

# Normalisation
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'\n✅ Données prêtes')
print(f'   Train : {X_train.shape}')
print(f'   Test  : {X_test.shape}')
print(f'   Features : {len(available_cols)}')

## 6. Benchmark des modèles

In [ ]:
# ── Définition des modèles ─────────────────────────────────────
MODELS = {
    'Ridge (baseline)': Ridge(alpha=1.0),
    'XGBoost': XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, verbosity=0
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        num_leaves=15, subsample=0.8,
        random_state=RANDOM_STATE, verbose=-1
    ),
    'CatBoost': CatBoostRegressor(
        iterations=200, depth=4, learning_rate=0.1,
        random_seed=RANDOM_STATE, verbose=0
    ),
}

# ── Benchmark ─────────────────────────────────────────────────
kf      = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

mlflow.set_experiment('MaroTrade_Benchmark')

print('=' * 65)
print(f'{"Modèle":<20} {"CV RMSE":>10} {"CV Std":>10} {"Test RMSE":>10} {"Test R²":>10} {"Temps":>8}')
print('=' * 65)

for name, model in MODELS.items():
    with mlflow.start_run(run_name=name):
        
        # Cross-validation
        cv_scores = cross_val_score(
            model, X_train, y_train,
            cv=kf, scoring='neg_root_mean_squared_error'
        )
        cv_rmse = -cv_scores.mean()
        cv_std  = cv_scores.std()
        
        # Entraînement final
        t0 = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - t0
        
        # Évaluation test
        y_pred = model.predict(X_test)
        rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
        mae    = mean_absolute_error(y_test, y_pred)
        r2     = r2_score(y_test, y_pred)
        
        # Logger MLflow
        mlflow.log_param('model', name)
        mlflow.log_metric('cv_rmse',    cv_rmse)
        mlflow.log_metric('test_rmse',  rmse)
        mlflow.log_metric('test_mae',   mae)
        mlflow.log_metric('test_r2',    r2)
        mlflow.log_metric('train_time', train_time)
        
        results[name] = {
            'CV RMSE':    round(cv_rmse, 4),
            'CV Std':     round(cv_std, 4),
            'Test RMSE':  round(rmse, 4),
            'Test MAE':   round(mae, 4),
            'Test R²':    round(r2, 4),
            'Temps (s)':  round(train_time, 3),
            'model_obj':  model
        }
        
        print(f'{name:<20} {cv_rmse:>10.4f} {cv_std:>10.4f} {rmse:>10.4f} {r2:>10.4f} {train_time:>7.2f}s')

print('=' * 65)

## 7. Visualisation des résultats

In [ ]:
# Tableau comparatif
df_results = pd.DataFrame({
    k: {m: v for m, v in v.items() if m != 'model_obj'}
    for k, v in results.items()
}).T.sort_values('Test RMSE')

print('\n📊 TABLEAU COMPARATIF (trié par Test RMSE croissant)')
print('=' * 65)
display(df_results)

In [ ]:
# ── Graphiques comparatifs ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

models_names = list(df_results.index)
colors = ['#E24B4A', '#378ADD', '#1D9E75', '#534AB7']

# RMSE
rmse_vals = df_results['Test RMSE'].astype(float)
bars = axes[0].bar(models_names, rmse_vals, color=colors[:len(models_names)])
axes[0].set_title('Test RMSE (↓ mieux)', fontsize=13)
axes[0].set_ylabel('RMSE')
for bar, val in zip(bars, rmse_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10)

# R²
r2_vals = df_results['Test R²'].astype(float)
bars = axes[1].bar(models_names, r2_vals, color=colors[:len(models_names)])
axes[1].set_title('Test R² (↑ mieux)', fontsize=13)
axes[1].set_ylabel('R²')
axes[1].set_ylim(0, 1.1)
for bar, val in zip(bars, r2_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10)

# Temps
time_vals = df_results['Temps (s)'].astype(float)
bars = axes[2].bar(models_names, time_vals, color=colors[:len(models_names)])
axes[2].set_title('Temps entraînement (s) (↓ mieux)', fontsize=13)
axes[2].set_ylabel('Secondes')
for bar, val in zip(bars, time_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}s', ha='center', va='bottom', fontsize=10)

plt.suptitle('Comparaison des modèles ML — MaroTrade Intelligence', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../data/raw/benchmark_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Prédictions vs réalité
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, (name, res) in enumerate(results.items()):
    ax  = axes[idx // 2][idx % 2]
    mdl = res['model_obj']
    y_p = mdl.predict(X_test)
    
    ax.scatter(y_test, y_p, alpha=0.5, s=20)
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Idéal')
    ax.set_xlabel('Valeurs réelles')
    ax.set_ylabel('Prédictions')
    ax.set_title(f'{name}\nRMSE={res["Test RMSE"]:.4f} | R²={res["Test R²"]:.4f}')
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.suptitle('Prédictions vs Réalité — Tous les modèles', fontsize=14)
plt.tight_layout()
plt.savefig('../data/raw/predictions_vs_reality.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Optimisation Optuna — Meilleur modèle

In [ ]:
# Identifier le meilleur modèle
best_name = df_results['Test RMSE'].astype(float).idxmin()
print(f'🏆 Meilleur modèle : {best_name}')
print(f'   Test RMSE : {df_results.loc[best_name, "Test RMSE"]}')
print(f'   Test R²   : {df_results.loc[best_name, "Test R²"]}')
print(f'\nLancement optimisation Optuna (100 essais)...')

In [ ]:
def objective_lgbm(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 500),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 10, 60),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': RANDOM_STATE, 'verbose': -1
    }
    model  = LGBMRegressor(**params)
    scores = cross_val_score(model, X_train, y_train,
                             cv=5, scoring='neg_root_mean_squared_error')
    return -scores.mean()

def objective_xgb(trial):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 100, 500),
        'max_depth':     trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':     trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':    trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': RANDOM_STATE, 'verbosity': 0
    }
    model  = XGBRegressor(**params)
    scores = cross_val_score(model, X_train, y_train,
                             cv=5, scoring='neg_root_mean_squared_error')
    return -scores.mean()

# Lancer Optuna
objectives = {'LightGBM': objective_lgbm, 'XGBoost': objective_xgb}
objective_fn = objectives.get(best_name, objective_lgbm)

study = optuna.create_study(direction='minimize')
study.optimize(objective_fn, n_trials=100, show_progress_bar=True)

print(f'\n✅ Optimisation terminée')
print(f'   Meilleurs paramètres : {study.best_params}')
print(f'   Meilleur CV RMSE     : {study.best_value:.4f}')

In [ ]:
# ── Modèle final optimisé ─────────────────────────────────────
if best_name == 'LightGBM':
    final_model = LGBMRegressor(**study.best_params, random_state=RANDOM_STATE, verbose=-1)
else:
    final_model = XGBRegressor(**study.best_params, random_state=RANDOM_STATE, verbosity=0)

# Entraîner sur toutes les données
final_model.fit(X_aug, y_aug)

# Évaluation finale
y_pred_final = final_model.predict(X_test)
rmse_final   = np.sqrt(mean_squared_error(y_test, y_pred_final))
r2_final     = r2_score(y_test, y_pred_final)

print(f'\n🏆 MODÈLE FINAL OPTIMISÉ ({best_name})')
print(f'   Test RMSE avant Optuna : {df_results.loc[best_name, "Test RMSE"]}')
print(f'   Test RMSE après Optuna : {rmse_final:.4f}')
print(f'   Test R²   après Optuna : {r2_final:.4f}')
print(f'   Amélioration RMSE      : {(float(df_results.loc[best_name, "Test RMSE"]) - rmse_final) / float(df_results.loc[best_name, "Test RMSE"]) * 100:.1f}%')

## 9. Analyse SHAP — Importance des features

In [ ]:
# Calcul SHAP
explainer   = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_test[:100])

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_test[:100],
    feature_names=available_cols,
    plot_type='bar',
    show=False
)
plt.title('Importance globale des features (SHAP)', fontsize=13)
plt.tight_layout()
plt.savefig('../data/raw/shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Feature importance numérique
shap_importance = pd.DataFrame({
    'feature':    available_cols,
    'importance': np.abs(shap_values).mean(axis=0)
}).sort_values('importance', ascending=False)

print('\n📊 Importance des features (SHAP) :')
display(shap_importance)

## 10. Sauvegarde du modèle final

In [ ]:
import joblib
from pathlib import Path

models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

# Sauvegarder modèle + scaler + features
joblib.dump(final_model, models_dir / 'best_scoring_model.pkl')
joblib.dump(scaler,      models_dir / 'feature_scaler.pkl')

model_config = {
    'model_name':     best_name,
    'feature_cols':   available_cols,
    'best_params':    study.best_params,
    'test_rmse':      rmse_final,
    'test_r2':        r2_final,
    'trained_on':     str(pd.Timestamp.now()),
    'n_samples':      len(X_aug),
    'n_features':     len(available_cols),
}

with open(models_dir / 'model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

# MLflow final
with mlflow.start_run(run_name=f'{best_name}_FINAL_OPTIMIZED'):
    mlflow.log_params(study.best_params)
    mlflow.log_metric('final_test_rmse', rmse_final)
    mlflow.log_metric('final_test_r2',   r2_final)
    mlflow.sklearn.log_model(final_model, 'best_model')

print('✅ Modèle sauvegardé :')
print(f'   models/best_scoring_model.pkl')
print(f'   models/feature_scaler.pkl')
print(f'   models/model_config.json')

## 11. Conclusion

In [ ]:
print('=' * 60)
print('  CONCLUSION BENCHMARK ML — MaroTrade Intelligence')
print('=' * 60)
print(f'\n  Données utilisées :')
print(f'    Produits       : {df_trade["hs_code"].nunique()}')
print(f'    Pays           : {df_trade["country_name"].nunique()}')
print(f'    Années         : 2015–2023')
print(f'    Lignes totales : {len(X_aug):,} (après augmentation ×5)')
print(f'    Features       : {len(available_cols)}')
print(f'\n  Résultats benchmark :')
for name, res in sorted(results.items(), key=lambda x: x[1]["Test RMSE"]):
    winner = ' ← GAGNANT' if name == best_name else ''
    print(f'    {name:<20} RMSE={res["Test RMSE"]:.4f}  R²={res["Test R²"]:.4f}  {res["Temps (s)"]:.2f}s{winner}')
print(f'\n  Modèle final ({best_name} + Optuna) :')
print(f'    Test RMSE : {rmse_final:.4f}')
print(f'    Test R²   : {r2_final:.4f}')
print(f'\n  Recommandation :')
print(f'    Utiliser {best_name} dans services/scoring/scoring_engine.py')
print(f'    Remplacer XGBRegressor par le modèle optimisé')
print('=' * 60)